# GrapePPI Model Training

Train the GrapePPI model using train.tsv and val.tsv. TSV format: protein1_ID, protein2_ID, label (0/1).

## 1. Dependencies and Utility Code

In [ ]:
# Google Colab environment setup (run first if using Colab)
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    # Set to your Drive folder containing data/ and models/, or "/content" if you upload files
    COLAB_PROJECT_ROOT = "/content/drive/MyDrive/grape-ppi"
    import os
    os.chdir(COLAB_PROJECT_ROOT)
    print(f"Colab: cwd = {os.getcwd()}")
else:
    COLAB_PROJECT_ROOT = None
    print("Running locally.")

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import h5py
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    roc_curve, auc, precision_recall_curve, average_precision_score,
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)
import matplotlib.pyplot as plt

plt.style.use('fivethirtyeight')


def set_seed(seed=1234):
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def calculate_metrics(y_true, y_pred):
    roc_auc = roc_auc_score(y_true, y_pred)
    pr_auc = average_precision_score(y_true, y_pred)
    y_pred_bin = (y_pred >= 0.5).astype(np.int32)
    acc = accuracy_score(y_true, y_pred_bin)
    precision = precision_score(y_true, y_pred_bin, average="binary", zero_division=0)
    recall = recall_score(y_true, y_pred_bin, average="binary", zero_division=0)
    f1 = f1_score(y_true, y_pred_bin, average="binary", zero_division=0)
    return acc, precision, recall, f1, roc_auc, pr_auc


def save_predictions(y_true, y_pred, save_path):
    os.makedirs(os.path.dirname(save_path) or '.', exist_ok=True)
    pd.DataFrame({"y_true": y_true, "y_pred": y_pred}).to_csv(save_path, index=False)


def pot_metrics(y_true, y_pred):
    """Plot ROC and PR curves (from util.py)"""
    fpr, tpr, _ = roc_curve(y_true, y_pred)
    roc_auc_val = auc(fpr, tpr)
    precision, recall, _ = precision_recall_curve(y_true, y_pred)
    pr_auc_val = average_precision_score(y_true, y_pred)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    ax1.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC (AUC = {roc_auc_val:.2f})')
    ax1.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random')
    ax1.set_xlim([0.0, 1.0]); ax1.set_ylim([0.0, 1.05])
    ax1.set_xlabel('False Positive Rate'); ax1.set_ylabel('True Positive Rate')
    ax1.set_title('ROC Curve'); ax1.legend(loc="lower right"); ax1.grid(True)
    ax2.plot(recall, precision, color='blue', lw=2, label=f'PR (AUC = {pr_auc_val:.2f})')
    ax2.axhline(y=np.sum(y_true)/len(y_true), color='r', linestyle='--', label='Random')
    ax2.set_xlim([0.0, 1.0]); ax2.set_ylim([0.0, 1.05])
    ax2.set_xlabel('Recall'); ax2.set_ylabel('Precision')
    ax2.set_title('Precision-Recall Curve'); ax2.legend(loc="upper right"); ax2.grid(True)
    plt.tight_layout(); plt.show()
    print(f"ROC AUC: {roc_auc_val:.4f}, PR AUC: {pr_auc_val:.4f}")

In [ ]:
# Data loading and model definition

class PooledDataset(Dataset):
    def __init__(self, names1, names2, labels, embedding_h5):
        self.names1 = names1
        self.names2 = names2
        self.labels = labels
        self.embed_data = {}
        names = set(names1).union(set(names2))
        with h5py.File(embedding_h5, "r") as h5fin:
            for name in names:
                self.embed_data[name] = torch.from_numpy(h5fin[name][:]).float()

    def __len__(self):
        return len(self.names1)

    def __getitem__(self, i):
        x1 = self.embed_data[self.names1[i]]
        x2 = self.embed_data[self.names2[i]]
        return x1, x2, torch.as_tensor(self.labels[i]).float()


def load_pooled_data(data_file, batch_size, embedding_h5, train=True):
    df = pd.read_csv(data_file, sep="\t", header=None)
    dataset = PooledDataset(df[0].to_list(), df[1].to_list(), df[2].to_list(), embedding_h5)
    return DataLoader(dataset, batch_size=batch_size, shuffle=train)


class GrapePPI(nn.Module):
    def __init__(self, esm_dim=1280, hidden_dim=512, dropout=0.2):
        super(GrapePPI, self).__init__()
        self.sequence_encoder = nn.Sequential(
            nn.Linear(esm_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
        )
        self.interaction_predictor = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, hidden_dim // 4),
            nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim // 4, 1),
        )

    def combine_features(self, x1, x2):
        return torch.cat([x1, x2], dim=-1)

    def forward(self, x1, x2):
        x1 = self.sequence_encoder(x1)
        x2 = self.sequence_encoder(x2)
        x = self.combine_features(x1, x2)
        logits = self.interaction_predictor(x)
        return F.sigmoid(logits).squeeze()

In [ ]:
def train_model(model, train_loader, val_loader, optimizer, scheduler=None,
                patience=20, epochs=100, device="cpu",
                model_save_path=None):
    best_val_loss = float("inf")
    patience_counter = 0
    history = {"train_loss": [], "val_loss": []}

    for epoch in range(1, epochs + 1):
        model.train()
        batch_losses = []
        for x1, x2, y in train_loader:
            optimizer.zero_grad()
            x1, x2, y = x1.to(device), x2.to(device), y.to(device)
            yhat = model(x1, x2)
            loss = F.binary_cross_entropy(yhat, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            batch_losses.append(loss.item())
        train_loss = np.mean(batch_losses)
        history["train_loss"].append(train_loss)

        model.eval()
        with torch.no_grad():
            y_true, y_pred, vlosses = [], [], []
            for x1, x2, y in val_loader:
                x1, x2, y = x1.to(device), x2.to(device), y.to(device)
                yhat = model(x1, x2)
                vlosses.append(F.binary_cross_entropy(yhat, y).item())
                y_true.append(y.cpu().numpy())
                y_pred.append(yhat.cpu().numpy())
            val_loss = np.mean(vlosses)
            history["val_loss"].append(val_loss)
            y_true = np.concatenate(y_true)
            y_pred = np.concatenate(y_pred)

        acc, precision, recall, f1, roc_auc, pr_auc = calculate_metrics(y_true, y_pred)
        lr = optimizer.param_groups[0]["lr"]
        msg = (f"Epoch: {epoch}, Loss: {train_loss:.4f}, Val_loss: {val_loss:.4f}, "
               f"Acc: {acc:.4f}, P: {precision:.4f}, R: {recall:.4f}, F1: {f1:.4f}, "
               f"ROC_AUC: {roc_auc:.4f}, PR_AUC: {pr_auc:.4f}, LR: {lr:.6f}")
        print(msg)

        if scheduler is not None:
            scheduler.step(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            if model_save_path:
                os.makedirs(os.path.dirname(model_save_path) or '.', exist_ok=True)
                torch.save({
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "val_loss": val_loss,
                    "metrics": {"accuracy": acc, "precision": precision, "recall": recall,
                               "f1": f1, "roc_auc": roc_auc, "pr_auc": pr_auc},
                    "esm_dim": getattr(model.sequence_encoder[0], 'in_features', 1280),
                }, model_save_path)
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping at epoch {epoch}")
                break
    
    return y_true, y_pred, history

## 2. Configuration and Data Paths

In [ ]:
set_seed(1234)

# Data directory: contains train.tsv, val.tsv, and ESM pooled embedding .h5 file
# For Grape-1 dataset
data_dir = "./data/Grape-1"   # Change to local path, e.g. "data" or absolute path
model_save_path = os.path.join("models", "grape_1_model.pt")

# For Grape-10 dataset
# data_dir = "./data/Grape-10"
# model_save_path = os.path.join("models", "grape_10_model.pt")

train_tsv = os.path.join(data_dir, "train.tsv")
val_tsv = os.path.join(data_dir, "val.tsv")
# ESM pooled embedding H5: keys are protein IDs, matching TSV columns 1 and 2; dimension must match esm_dim
embedding_h5 = os.path.join(data_dir, "esm2_t36_3B.h5")
batch_size = 128
epochs = 100
patience = 20
lr = 1e-3
esm_dim = 2560   # Must match embedding_h5 dimension: esm2_t36_3B -> 2560
hidden_dim = 512
dropout = 0.2

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
print(f"Train: {train_tsv}, Val: {val_tsv}")
print(f"Embedding: {embedding_h5}, esm_dim={esm_dim}")

## 3. Load Data and Train

In [ ]:
train_loader = load_pooled_data(train_tsv, batch_size, embedding_h5, train=True)
val_loader = load_pooled_data(val_tsv, batch_size, embedding_h5, train=False)
print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

model = GrapePPI(esm_dim=esm_dim, hidden_dim=hidden_dim, dropout=dropout).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=10, min_lr=lr * 0.01
)

y_true, y_pred, history = train_model(
    model, train_loader, val_loader, optimizer, scheduler=scheduler,
    patience=patience, epochs=epochs, device=device,
    model_save_path=model_save_path
)

## 3.1 Best Saved Model Parameters

In [ ]:
# Load the best saved checkpoint and display its metadata and parameter counts
ckpt = torch.load(model_save_path, map_location="cpu", weights_only=False)
print("Best saved checkpoint:")
print(f"  Path: {model_save_path}")
print(f"  Epoch: {ckpt.get('epoch', '?')}")
print(f"  Val loss: {ckpt.get('val_loss', 0.):.4f}")
metrics = ckpt.get("metrics", {})
if metrics:
    print("  Metrics:", ", ".join(f"{k}={v:.4f}" for k, v in metrics.items()))
esm_dim_ckpt = ckpt.get("esm_dim", 2560)

# Build model from state_dict to report parameter counts
best_model = GrapePPI(esm_dim=esm_dim_ckpt, hidden_dim=hidden_dim, dropout=dropout)
best_model.load_state_dict(ckpt["model_state_dict"], strict=True)
total_params = sum(p.numel() for p in best_model.parameters())
trainable_params = sum(p.numel() for p in best_model.parameters() if p.requires_grad)
print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print("Parameters by layer:")
for name, param in best_model.named_parameters():
    print(f"  {name}: {param.numel():,}  shape={tuple(param.shape)}")

## 3.2 Training Loss Curves

In [ ]:
# Plot training and validation loss
epochs_range = range(1, len(history["train_loss"]) + 1)
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(epochs_range, history["train_loss"], "b-", label="Train Loss", lw=2)
ax.plot(epochs_range, history["val_loss"], "r-", label="Val Loss", lw=2)
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("Training and Validation Loss")
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

## 4. Validation Set ROC/PR Curves

In [ ]:
pot_metrics(y_true, y_pred)